In [10]:
import pandas as pd
import numpy as np
import openpyxl 
import os

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# سنسورهای هدف
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_analysis():
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
        return

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print("✅ مرحله ۱: فایل بارگذاری شد.")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return

    # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر (Fault)
    try:
        last_date = df.index.max()
        split_date = last_date - pd.Timedelta(days=30)
        baseline_start = split_date - pd.Timedelta(days=30)

        df_baseline = df.loc[baseline_start:split_date].copy()
        df_fault = df.loc[split_date:last_date].copy()
    except Exception as e:
        print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
        return

    # ۲. محاسبه شیب و TTT برای یک ماه اخیر
    print("⏳ در حال محاسبه شیب تغییرات و زمان تخمینی برای سنسورهای هدف...")

    target_analysis_results = []

    for col in target_sensors:
        if col not in df_fault.columns: continue

        try:
            # الف) حد آستانه از بازه سلامت
            mean_base = df_baseline[col].mean()
            std_base = df_baseline[col].std()
            upper_limit = mean_base + 3 * std_base 

            # ب) محاسبه شیب (تغییرات در هر ساعت)
            time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
            val_diff = df_fault[col].diff()

            instant_slopes = val_diff / time_diff_series
            avg_slope = instant_slopes.mean()

            current_val = df_fault[col].iloc[-1] 

            # ج) محاسبه TTT (ساعت مانده تا حد بحرانی)
            ttt_hours = np.nan
            if avg_slope > 0 and current_val < upper_limit:
                ttt_hours = (upper_limit - current_val) / avg_slope

            target_analysis_results.append({
                'Sensor': col,
                'Current_Value': round(current_val, 4),
                'Baseline_Limit(3Sigma)': round(upper_limit, 4),
                'Average_Slope_per_Hour': round(avg_slope, 6),
                'Hours_to_Threshold': round(ttt_hours, 2) if (not np.isnan(ttt_hours) and ttt_hours != np.inf) else "No Risk"
            })
        except: continue

    # ۳. رتبه‌بندی RCA
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        try:
            dev = abs(df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() + 1e-6)
            rca_list.append({'Sensor': c, 'Deviation_Score': round(dev, 4)})
        except: continue
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)

    # ۴. ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)

        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)

        print(f"🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

# فراخوانی مستقیم تابع برای جلوگیری از خطای NameError
run_analysis()

✅ مرحله ۱: فایل بارگذاری شد.
⏳ در حال محاسبه شیب تغییرات و زمان تخمینی برای سنسورهای هدف...
🚀 گزارش با موفقیت ساخته شد:
outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output2.xlsx
